# Food Delivery Data Analysis

Innomatics Research Labs – GenAI Internship Entrance Test

## Import Libraries

In [71]:
import pandas as pd

orders = pd.read_csv("orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [72]:
users = pd.read_json("users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [ ]:
import sqlite3
import pandas as pd


conn = sqlite3.connect(":memory:")


with open("restaurants.sql", "r") as f:
    conn.executescript(f.read())


restaurants = pd.read_sql("SELECT * FROM restaurants", conn)

restaurants.head()




,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [22]:
final_df = (
    orders
    .merge(users, on="user_id", how="left")
    .merge(restaurants, on="restaurant_id", how="left")
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [23]:
df = orders.merge(
    users,
    on="user_id",
    how="left"
)


In [24]:
final_df = df.merge(
    restaurants,
    on="restaurant_id",
    how="left"
)


In [25]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [26]:
import os
os.getcwd()



'd:\\Downloads'


Which city has the highest total revenue from Gold members?

In [32]:
gold_df = final_df[final_df["membership"] == "Gold"]
gold_df.groupby("city")["total_amount"].sum()


city
Bangalore     994702.59
Chennai      1080909.79
Hyderabad     896740.19
Pune         1003012.32
Name: total_amount, dtype: float64

In [ ]:
# Q2: Cuisine with highest average order value
q2 = (
    final_df.groupby("cuisine")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

q2








C:\Users\adars\AppData\Local\Temp\ipykernel_12092\250710837.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby("rating_range")["total_amount"]


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [43]:
# Q3: Users with total spend > 1000
user_spend = final_df.groupby("user_id")["total_amount"].sum()

q3 = user_spend[user_spend > 1000].count()
q3


np.int64(2544)

In [45]:
# Q4: Rating range with highest revenue
import pandas as pd

bins = [0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0–3.5", "3.6–4.0", "4.1–4.5", "4.6–5.0"]

final_df["rating_range"] = pd.cut(final_df["rating"], bins=bins, labels=labels)

q4 = (
    final_df.groupby("rating_range")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

q4

C:\Users\adars\AppData\Local\Temp\ipykernel_12092\2263448471.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby("rating_range")["total_amount"]


rating_range
4.6–5.0    2197030.75
3.0–3.5    2136772.70
4.1–4.5    1960326.26
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [46]:
# Q5: Gold members - highest average order value city
q5 = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)

q5

city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [57]:
# Q6: Cuisine with lowest restaurants but high revenue
restaurant_count = final_df.groupby("cuisine")["restaurant_id"].nunique()
revenue = final_df.groupby("cuisine")["total_amount"].sum()

q6 = pd.concat([restaurant_count, revenue], axis=1)
q6.columns = ["restaurant_count", "revenue"]

q6.sort_values("restaurant_count")


,restaurant_count,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [56]:
# Q7: Percentage of orders by Gold members
q7 = (
    final_df[final_df["membership"] == "Gold"].shape[0]
    / final_df.shape[0]
) * 100

round(q7)

50

In [55]:
# Q8: Highest AOV restaurant with < 20 orders

# q8 = (
#     final_df
#     .groupby("restaurant_name_y")
#     .agg(
#         orders=("order_id", "count"),
#         avg_order_value=("total_amount", "mean")
#     )
# )

# q8[q8["orders"] < 20] \
#     .sort_values("avg_order_value", ascending=False) \
#     .head(1)


q8 = (
    final_df
    .groupby(["restaurant_id", "restaurant_name_y"])
    .agg(
        orders=("order_id", "count"),
        avg_order_value=("total_amount", "mean")
    )
)

q8[q8["orders"] < 20] \
    .sort_values("avg_order_value", ascending=False) \
    .head(1)



,,orders,avg_order_value
restaurant_id,restaurant_name_y,,
294,Restaurant_294,13,1040.222308


In [52]:
# Q9: Membership + Cuisine highest revenue
q9 = (
    final_df.groupby(["membership", "cuisine"])["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

q9


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [42]:
# Q10: Quarter with highest revenue
final_df["order_date"] = pd.to_datetime(final_df["order_date"], dayfirst=True)
final_df["quarter"] = final_df["order_date"].dt.to_period("Q")

q10 = (
    final_df.groupby("quarter")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

q10

quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [58]:
# How many total orders were placed by users with Gold membership?
final_df[final_df["membership"] == "Gold"]["order_id"].count()


np.int64(4987)

In [60]:
# What is the total revenue (rounded to nearest integer) generated from orders placed in Hyderabad city?
round(
    final_df[final_df["city"] == "Hyderabad"]["total_amount"].sum()
)

1889367

In [61]:
# How many distinct users placed at least one order?
final_df["user_id"].nunique()

2883

In [62]:
# Average order value for Gold members (2 decimals)
round(
    final_df[final_df["membership"] == "Gold"]["total_amount"].mean(),
    2
)


np.float64(797.15)

In [63]:
# Orders placed for restaurants with rating >= 4.5
final_df[final_df["rating"] >= 4.5]["order_id"].count()


np.int64(3374)

In [64]:
# How many orders were placed in the top revenue city among Gold members only?
top_city = (
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
    .index[0]
)
top_city
final_df[
    (final_df["membership"] == "Gold") &
    (final_df["city"] == top_city)
]["order_id"].count()


np.int64(1337)

In [65]:
# The column used to join orders.csv and users.json is ________.
orders.merge(users, on="user_id", how="left")


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular
...,...,...,...,...,...,...,...,...,...
9995,9996,2528,249,21-05-2023,1211.96,Royal Kitchen North Indian,User_2528,Hyderabad,Gold
9996,9997,2867,267,06-08-2023,1188.05,Darbar Cafe Punjabi,User_2867,Bangalore,Regular
9997,9998,522,420,11-11-2023,979.44,Ruchi Tiffins Chinese,User_522,Bangalore,Gold
9998,9999,319,492,08-09-2023,1105.93,Swagath Kitchen North Indian,User_319,Bangalore,Gold


In [66]:
# The dataset containing cuisine and rating information is stored in ________ format.
open("restaurants.sql")

<_io.TextIOWrapper name='restaurants.sql' mode='r' encoding='UTF-8'>

In [67]:
# The total number of rows in the final merged dataset is ________.
final_df.shape[0]


10000

In [68]:
# If a user has no matching record in users.json, the merged values will be ________.
how="left"